# AI StudyMate — RAG Pipeline (Semantic Chunking Rework)
 (load → chunk → embed → hybrid retrieve → generate → evaluate)

In [1]:
%pip install -q pymupdf chromadb sentence-transformers ollama tiktoken pandas rank_bm25 numpy


Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import re
import statistics
from collections import Counter
from pathlib import Path

import pandas as pd
import numpy as np
import pymupdf
import tiktoken
import chromadb
import ollama
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PDF_DIR = PROJECT_ROOT / "data" / "raw_pdfs"
METADATA_DIR = PROJECT_ROOT / "data" / "metadata"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
VECTOR_STORE_DIR = PROJECT_ROOT / "data" / "vector_store"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
OLLAMA_MODEL_NAME = "llama3.2:3b"
TOKENIZER_NAME = "cl100k_base"
TARGET_CHUNK_TOKENS = 800
OVERLAP_TOKENS = 120
MIN_CHUNK_TOKENS = 150
SOURCE_POOL_SIZE = 20
FUSED_CANDIDATE_POOL = 12
FINAL_TOP_K = 5
RRF_K = 60
COLLECTION_NAME = "ai_studymate_ml_dl"
LLM_TEMPERATURE = 0
REFUSAL_TEXT = "I don't have enough information in my verified sources to answer this question."
OUT_OF_SCOPE_TEXT = "I only answer questions about Artificial Intelligence, Machine Learning, and Deep Learning."

# --- new: semantic chunking knobs ---
SEMANTIC_BREAKPOINT_PERCENTILE = 90   # how sharp a meaning-shift has to be (vs. the rest
                                       # of the section) before we cut a new segment there
DROP_FRAGMENT_TOKENS = 20             # true junk fragments (stray numbers, running
                                       # headers) get dropped instead of contaminating a neighbor

tokenizer = tiktoken.get_encoding(TOKENIZER_NAME)


c:\Users\omara\anaconda3\envs\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2.1 Load & Inspect

In [3]:
def load_sources_metadata():
    with open(METADATA_DIR / "sources.json") as f:
        return json.load(f)

sources_metadata = load_sources_metadata()
pdf_sources = [
    s for s in sources_metadata["pdf_corpus_sources"]
    if s["status"] in ("downloaded", "skipped_existing")
]
len(pdf_sources)


19

In [4]:
def extract_pages_with_layout(pdf_path):
    doc = pymupdf.open(str(pdf_path))
    pages = []
    for page in doc:
        text_dict = page.get_text("dict")
        lines = []
        for block in text_dict["blocks"]:
            if "lines" not in block:
                continue
            for line in block["lines"]:
                spans = line.get("spans", [])
                if not spans:
                    continue
                line_text = "".join(s["text"] for s in spans).strip()
                if not line_text:
                    continue
                max_size = max(s["size"] for s in spans)
                is_bold = any("bold" in s.get("font", "").lower() for s in spans)
                lines.append({
                    "text": line_text,
                    "size": round(max_size, 1),
                    "bold": is_bold,
                    "y0": line["bbox"][1],
                })
        lines.sort(key=lambda l: l["y0"])
        pages.append(lines)
    doc.close()
    return pages

source_pages_layout = {}
inspection_records = []

for source in pdf_sources:
    pdf_path = PROJECT_ROOT / source["local_path"]
    try:
        pages = extract_pages_with_layout(pdf_path)
        source_pages_layout[source["id"]] = pages
        total_chars = sum(sum(len(l["text"]) for l in page) for page in pages)
        inspection_records.append({
            "id": source["id"],
            "title": source["title"],
            "num_pages": len(pages),
            "total_chars": total_chars,
            "avg_chars_per_page": round(total_chars / max(len(pages), 1), 1),
            "parse_ok": total_chars > 200,
        })
    except Exception:
        source_pages_layout[source["id"]] = []
        inspection_records.append({
            "id": source["id"],
            "title": source["title"],
            "num_pages": 0,
            "total_chars": 0,
            "avg_chars_per_page": 0,
            "parse_ok": False,
        })

inspection_df = pd.DataFrame(inspection_records)
inspection_df


,id,title,num_pages,total_chars,avg_chars_per_page,parse_ok
0,attention_is_all_you_need,Attention Is All You Need,15,38450,2563.3,True
1,adam_optimizer,Adam: A Method for Stochastic Optimization,15,40475,2698.3,True
2,dropout,Dropout: A Simple Way to Prevent Neural Networ...,30,76798,2559.9,True
3,batch_normalization,Batch Normalization: Accelerating Deep Network...,11,44185,4016.8,True
4,resnet,Deep Residual Learning for Image Recognition,12,57639,4803.2,True
5,lstm_search_space_odyssey,LSTM: A Search Space Odyssey,12,56993,4749.4,True
6,gan,Generative Adversarial Networks,9,28444,3160.4,True
7,vae,Auto-Encoding Variational Bayes,14,40429,2887.8,True
8,ddpm,Denoising Diffusion Probabilistic Models,25,52768,2110.7,True
9,random_forests,Random Forests,33,57451,1740.9,True


In [5]:
failed_sources_df = inspection_df[inspection_df["parse_ok"] == False]
failed_sources_df


,id,title,num_pages,total_chars,avg_chars_per_page,parse_ok


## 2.2 Chunking Strategy (Structure-Aware + Semantic)

In [6]:
# Loaded here (rather than in 2.3) because semantic segmentation needs sentence
# embeddings at chunk-build time, not just at index time.
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6867.71it/s]


In [7]:
NUMBERED_HEADING_PATTERN = re.compile(r"^\d{1,2}(\.\d{1,2}){0,2}\.?\s+[A-Z].{2,90}$")
COMMON_UNNUMBERED_HEADINGS = {
    "abstract", "introduction", "references", "acknowledgments",
    "acknowledgements", "related work", "conclusion", "conclusions",
    "background", "appendix", "discussion", "results", "methodology", "methods",
}
HEADING_FONT_RATIO = 1.15
BOLD_HEADING_FONT_RATIO = 1.05
UNTITLED_SECTION_LABEL = "Untitled Section"

def compute_body_font_size(lines):
    sizes = [l["size"] for l in lines if len(l["text"].split()) > 4]
    if not sizes:
        sizes = [l["size"] for l in lines]
    if not sizes:
        return 10.0
    return Counter(sizes).most_common(1)[0][0]

def is_heading_line(line, body_size):
    text = line["text"].strip()
    if not text or len(text) > 100:
        return False
    words = text.split()
    if len(words) == 0 or len(words) > 12:
        return False
    if NUMBERED_HEADING_PATTERN.match(text):
        return True
    normalized = text.strip(":").lower()
    if normalized in COMMON_UNNUMBERED_HEADINGS:
        return True
    ends_with_sentence_punct = text.endswith((".", ",", ";"))
    if not ends_with_sentence_punct and line["size"] >= body_size * HEADING_FONT_RATIO:
        return True
    if not ends_with_sentence_punct and line["bold"] and line["size"] >= body_size * BOLD_HEADING_FONT_RATIO:
        return True
    return False

def split_page_into_sections(lines, body_size):
    """Layout-based structural signal -- still useful as a coarse boundary
    (we never let a merge cross one of these), just no longer the *only*
    signal deciding where a chunk starts and ends."""
    sections = []
    current_title = UNTITLED_SECTION_LABEL
    current_lines = []
    for line in lines:
        text = line["text"].strip()
        if re.fullmatch(r"\d{1,4}", text):
            continue
        if is_heading_line(line, body_size):
            if current_lines:
                sections.append((current_title, current_lines))
            current_title = text
            current_lines = []
        else:
            current_lines.append(line)
    if current_lines:
        sections.append((current_title, current_lines))
    return sections


In [8]:
def split_into_sentences(text):
    sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", text)
    return [s.strip() for s in sentences if s.strip()]

def cosine_similarity(vector_a, vector_b):
    a = np.asarray(vector_a)
    b = np.asarray(vector_b)
    denom = (np.linalg.norm(a) * np.linalg.norm(b)) + 1e-8
    return float(np.dot(a, b) / denom)


In [9]:
def semantic_segment_sentences(sentences, percentile=SEMANTIC_BREAKPOINT_PERCENTILE):
    """
    Group a list of sentences (already in reading order, from ONE structural
    section) into semantically coherent segments.

    Method: embed every sentence, measure how much the meaning shifts from
    one sentence to the next (1 - cosine similarity), then cut wherever that
    shift is unusually large relative to the rest of the section (top
    `percentile`). This is what actually finds topic boundaries -- font size
    and line spacing (the old signal) can't tell "definition of precision"
    apart from "definition of recall" if they happen to be typeset
    identically, but the embeddings can.
    """
    if len(sentences) <= 2:
        return [sentences] if sentences else []

    embeddings = embedding_model.encode(sentences, show_progress_bar=False)
    shifts = [
        1.0 - cosine_similarity(embeddings[i - 1], embeddings[i])
        for i in range(1, len(embeddings))
    ]
    threshold = float(np.percentile(shifts, percentile))

    segments, current = [], [sentences[0]]
    for i, shift in enumerate(shifts, start=1):
        if shift > threshold:
            segments.append(current)
            current = [sentences[i]]
        else:
            current.append(sentences[i])
    segments.append(current)
    return segments


In [10]:
def split_long_sentence(sentence, max_tokens):
    """Fallback for the rare oversized "sentence" (an unbroken equation block,
    a run-on reference list, etc). Tries to break on clause punctuation
    (commas/semicolons) first, so pieces still end at a natural pause.
    Raw token-index slicing is only used as a last resort, on whatever is
    still left over after clause splitting -- never as the first move."""
    clauses = re.split(r"(?<=[;,])\s+", sentence)
    pieces, current = [], ""
    for clause in clauses:
        candidate = (current + " " + clause).strip() if current else clause
        if current and len(tokenizer.encode(candidate)) > max_tokens:
            pieces.append(current)
            current = clause
        else:
            current = candidate
    if current:
        pieces.append(current)

    final_pieces = []
    for piece in pieces:
        ids = tokenizer.encode(piece)
        if len(ids) <= max_tokens:
            final_pieces.append(piece)
        else:
            for start in range(0, len(ids), max_tokens):
                final_pieces.append(tokenizer.decode(ids[start:start + max_tokens]))
    return final_pieces


def pack_semantic_segments(segments, max_tokens=TARGET_CHUNK_TOKENS, overlap_tokens=OVERLAP_TOKENS):
    """Pack semantically-coherent sentence segments into token-bounded chunks.
    Never cuts inside a sentence except via split_long_sentence's clause-aware
    fallback. Overlap is carried forward as whole trailing sentences (not a
    raw token slice), so a chunk's lead-in always starts at a sentence
    boundary instead of mid-word."""
    chunks = []
    current_sentences, current_tokens = [], 0

    def sentence_tokens(s):
        return len(tokenizer.encode(s))

    def flush(carry_overlap=True):
        nonlocal current_sentences, current_tokens
        if not current_sentences:
            return
        chunks.append(" ".join(current_sentences))
        if not carry_overlap:
            current_sentences, current_tokens = [], 0
            return
        overlap_sents, tok_budget = [], overlap_tokens
        for s in reversed(current_sentences):
            t = sentence_tokens(s)
            if t > tok_budget and overlap_sents:
                break
            overlap_sents.insert(0, s)
            tok_budget -= t
            if tok_budget <= 0:
                break
        current_sentences = overlap_sents
        current_tokens = sum(sentence_tokens(s) for s in current_sentences)

    for segment in segments:
        for sentence in segment:
            s_tokens = sentence_tokens(sentence)
            pieces = split_long_sentence(sentence, max_tokens) if s_tokens > max_tokens else [sentence]
            for piece in pieces:
                p_tokens = sentence_tokens(piece)
                if current_tokens + p_tokens > max_tokens and current_sentences:
                    flush()
                current_sentences.append(piece)
                current_tokens += p_tokens

    flush(carry_overlap=False)
    return [c for c in chunks if c.strip()]


In [11]:
def combine_chunk_records(a, b):
    combined_text = a["text"] + " " + b["text"]
    page_label = a["page"] if a["page"] == b["page"] else f"{a['page']}-{b['page']}"
    return {
        "chunk_id": a["chunk_id"],
        "source_id": a["source_id"],
        "title": a["title"],
        "authors": a["authors"],
        "page": page_label,
        "section": a["section"],  # always equal by construction -- see the guard in merge_small_chunks
        "source_url": a["source_url"],
        "topic": a["topic"],
        "text": combined_text,
        "token_count": len(tokenizer.encode(combined_text)),
    }

def merge_small_chunks(chunk_records, min_tokens=MIN_CHUNK_TOKENS, drop_below_tokens=DROP_FRAGMENT_TOKENS):
    """
    Unlike the old version, this NEVER merges across a section boundary and
    NEVER merges across a page gap bigger than 1. A too-small but topically
    pure chunk is kept as-is rather than glued to unrelated content -- that
    blind gluing is exactly what produced chunks like
    "Untitled Section / Isabelle Guyon / Abstract" in the old pipeline, where
    the retrieved chunk was only half about the question and the LLM had
    nothing clean to cite.

    Genuine junk fragments (stray page numbers, running headers that slipped
    through) are dropped outright instead of being merged into a neighbor.
    """
    merged = []
    buffer = None
    for record in chunk_records:
        if record["token_count"] < drop_below_tokens:
            continue
        if buffer is None:
            buffer = record
            continue
        same_section = buffer["section"] == record["section"]
        try:
            page_gap = abs(int(record["page"].split("-")[0]) - int(buffer["page"].split("-")[-1]))
        except ValueError:
            page_gap = 99
        can_merge = buffer["token_count"] < min_tokens and same_section and page_gap <= 1
        if can_merge:
            buffer = combine_chunk_records(buffer, record)
        else:
            merged.append(buffer)
            buffer = record
    if buffer is not None:
        merged.append(buffer)
    return merged


In [12]:
def build_chunks_for_source(source):
    pages = source_pages_layout.get(source["id"], [])
    raw_records = []
    chunk_index = 0
    for page_number, lines in enumerate(pages, start=1):
        if not lines:
            continue
        body_size = compute_body_font_size(lines)
        sections = split_page_into_sections(lines, body_size)
        for section_title, section_lines in sections:
            section_text = " ".join(l["text"] for l in section_lines).strip()
            if not section_text:
                continue
            sentences = split_into_sentences(section_text)
            if not sentences:
                continue
            segments = semantic_segment_sentences(sentences)
            packed = pack_semantic_segments(segments)
            for chunk_text_value in packed:
                if len(chunk_text_value.strip()) < 20:
                    continue
                raw_records.append({
                    "chunk_id": f"{source['id']}_p{page_number}_c{chunk_index}",
                    "source_id": source["id"],
                    "title": source["title"],
                    "authors": source["authors"],
                    "page": str(page_number),
                    "section": section_title,
                    "source_url": source["url"],
                    "topic": source["topic"],
                    "text": chunk_text_value,
                    "token_count": len(tokenizer.encode(chunk_text_value)),
                })
                chunk_index += 1
    return merge_small_chunks(raw_records)

all_chunks = []
for source in pdf_sources:
    all_chunks.extend(build_chunks_for_source(source))

len(all_chunks)


654

In [13]:
chunks_preview_df = pd.DataFrame(all_chunks).drop(columns=["text"]).head(20)
chunks_preview_df


,chunk_id,source_id,title,authors,page,section,source_url,topic,token_count
0,attention_is_all_you_need_p1_c1,attention_is_all_you_need,Attention Is All You Need,"Vaswani, Shazeer, Parmar, Uszkoreit, Jones, Go...",1,arXiv:1706.03762v7 [cs.CL] 2 Aug 2023,https://arxiv.org/pdf/1706.03762,"Attention mechanisms, Transformers",123
1,attention_is_all_you_need_p1_c2,attention_is_all_you_need,Attention Is All You Need,"Vaswani, Shazeer, Parmar, Uszkoreit, Jones, Go...",1,Abstract,https://arxiv.org/pdf/1706.03762,"Attention mechanisms, Transformers",428
2,attention_is_all_you_need_p2_c3,attention_is_all_you_need,Attention Is All You Need,"Vaswani, Shazeer, Parmar, Uszkoreit, Jones, Go...",2,Introduction,https://arxiv.org/pdf/1706.03762,"Attention mechanisms, Transformers",341
3,attention_is_all_you_need_p2_c4,attention_is_all_you_need,Attention Is All You Need,"Vaswani, Shazeer, Parmar, Uszkoreit, Jones, Go...",2,Background,https://arxiv.org/pdf/1706.03762,"Attention mechanisms, Transformers",344
4,attention_is_all_you_need_p2_c5,attention_is_all_you_need,Attention Is All You Need,"Vaswani, Shazeer, Parmar, Uszkoreit, Jones, Go...",2,Model Architecture,https://arxiv.org/pdf/1706.03762,"Attention mechanisms, Transformers",106
5,attention_is_all_you_need_p3_c6,attention_is_all_you_need,Attention Is All You Need,"Vaswani, Shazeer, Parmar, Uszkoreit, Jones, Go...",3,Untitled Section,https://arxiv.org/pdf/1706.03762,"Attention mechanisms, Transformers",378
6,attention_is_all_you_need_p4_c7,attention_is_all_you_need,Attention Is All You Need,"Vaswani, Shazeer, Parmar, Uszkoreit, Jones, Go...",4,Untitled Section,https://arxiv.org/pdf/1706.03762,"Attention mechanisms, Transformers",518
7,attention_is_all_you_need_p5_c8,attention_is_all_you_need,Attention Is All You Need,"Vaswani, Shazeer, Parmar, Uszkoreit, Jones, Go...",5,Untitled Section,https://arxiv.org/pdf/1706.03762,"Attention mechanisms, Transformers",700
8,attention_is_all_you_need_p6_c9,attention_is_all_you_need,Attention Is All You Need,"Vaswani, Shazeer, Parmar, Uszkoreit, Jones, Go...",6,Untitled Section,https://arxiv.org/pdf/1706.03762,"Attention mechanisms, Transformers",444
9,attention_is_all_you_need_p6_c10,attention_is_all_you_need,Attention Is All You Need,"Vaswani, Shazeer, Parmar, Uszkoreit, Jones, Go...",6,Why Self-Attention,https://arxiv.org/pdf/1706.03762,"Attention mechanisms, Transformers",284


In [14]:
untitled_section_ratio = sum(1 for c in all_chunks if c["section"] == UNTITLED_SECTION_LABEL) / len(all_chunks)
cross_section_chunk_count = sum(1 for c in all_chunks if " / " in c["section"])  # should be 0 now
token_count_series = pd.Series([c["token_count"] for c in all_chunks])
chunking_stats_df = pd.DataFrame([{
    "total_chunks": len(all_chunks),
    "mean_tokens": round(token_count_series.mean(), 1),
    "median_tokens": round(token_count_series.median(), 1),
    "min_tokens": int(token_count_series.min()),
    "max_tokens": int(token_count_series.max()),
    "untitled_section_ratio": round(untitled_section_ratio, 3),
    "cross_section_merged_chunks": cross_section_chunk_count,
}])
chunking_stats_df


,total_chunks,mean_tokens,median_tokens,min_tokens,max_tokens,untitled_section_ratio,cross_section_merged_chunks
0,654,335.3,292.0,20,931,0.472,0


## 2.3 Embeddings & Vector Store

In [15]:
# embedding_model was already loaded at the top of 2.2 (semantic chunking needs
# it at chunk-build time), so we just reuse it here for indexing.
chunk_ids = [c["chunk_id"] for c in all_chunks]
chunk_texts = [c["text"] for c in all_chunks]
chunk_id_to_chunk = {c["chunk_id"]: c for c in all_chunks}
chunk_metadatas = [
    {
        "source_id": c["source_id"],
        "title": c["title"],
        "authors": c["authors"],
        "page": c["page"],
        "section": c["section"],
        "source_url": c["source_url"],
        "topic": c["topic"],
    }
    for c in all_chunks
]

chunk_embeddings = embedding_model.encode(
    chunk_texts, show_progress_bar=True, batch_size=32
).tolist()


Batches: 100%|██████████| 21/21 [00:13<00:00,  1.57it/s]


In [16]:
chroma_client = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR))

try:
    chroma_client.delete_collection(name=COLLECTION_NAME)
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={
        "embedding_model": EMBEDDING_MODEL_NAME,
        "tokenizer": TOKENIZER_NAME,
        "chunk_size_tokens": TARGET_CHUNK_TOKENS,
        "chunk_overlap_tokens": OVERLAP_TOKENS,
        "chunking_method": "structure_aware_semantic",
    },
)

BATCH_SIZE = 100
for start in range(0, len(chunk_ids), BATCH_SIZE):
    end = start + BATCH_SIZE
    collection.add(
        ids=chunk_ids[start:end],
        embeddings=chunk_embeddings[start:end],
        documents=chunk_texts[start:end],
        metadatas=chunk_metadatas[start:end],
    )

collection.count()


654

In [17]:
def tokenize_for_bm25(text):
    return re.findall(r"[a-zA-Z0-9]+", text.lower())

bm25_corpus_tokens = [tokenize_for_bm25(text) for text in chunk_texts]
bm25_index = BM25Okapi(bm25_corpus_tokens)
len(bm25_corpus_tokens)


654

In [18]:
chunk_id_to_embedding = dict(zip(chunk_ids, chunk_embeddings))
len(chunk_id_to_embedding)


654

## 2.4 Retrieval & Prompting

In [19]:
def dense_search_ids(query, n=SOURCE_POOL_SIZE):
    query_embedding = embedding_model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=n)
    return results["ids"][0]

def bm25_search_ids(query, n=SOURCE_POOL_SIZE):
    query_tokens = tokenize_for_bm25(query)
    scores = bm25_index.get_scores(query_tokens)
    ranked_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:n]
    return [chunk_ids[i] for i in ranked_indices]

def reciprocal_rank_fusion(ranked_id_lists, k=RRF_K):
    fused_scores = {}
    for ranked_ids in ranked_id_lists:
        for rank, doc_id in enumerate(ranked_ids):
            fused_scores[doc_id] = fused_scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    return sorted(fused_scores.items(), key=lambda item: item[1], reverse=True)


In [20]:
def rerank_by_cosine_similarity(query, candidate_ids, top_k):
    query_embedding = embedding_model.encode([query])[0]
    scored = []
    for doc_id in candidate_ids:
        similarity = cosine_similarity(query_embedding, chunk_id_to_embedding[doc_id])
        scored.append((doc_id, similarity))
    scored.sort(key=lambda item: item[1], reverse=True)
    return [doc_id for doc_id, similarity in scored[:top_k]]

def hybrid_retrieve(query, top_k=FINAL_TOP_K, source_pool=SOURCE_POOL_SIZE, fused_pool=FUSED_CANDIDATE_POOL):
    dense_ids = dense_search_ids(query, n=source_pool)
    sparse_ids = bm25_search_ids(query, n=source_pool)
    fused = reciprocal_rank_fusion([dense_ids, sparse_ids])
    candidate_ids = [doc_id for doc_id, score in fused[:fused_pool]]
    reranked_ids = rerank_by_cosine_similarity(query, candidate_ids, top_k)
    retrieved = []
    for doc_id in reranked_ids:
        chunk = chunk_id_to_chunk[doc_id]
        retrieved.append({
            "chunk_id": doc_id,
            "text": chunk["text"],
            "metadata": {
                "source_id": chunk["source_id"],
                "title": chunk["title"],
                "authors": chunk["authors"],
                "page": chunk["page"],
                "section": chunk["section"],
                "source_url": chunk["source_url"],
                "topic": chunk["topic"],
            },
        })
    return retrieved


In [21]:
CHITCHAT_PATTERNS = [
    re.compile(r"^\s*(hi|hello|hey|yo)\s*[!.]*\s*$", re.IGNORECASE),
    re.compile(r"^\s*good\s+(morning|afternoon|evening)\s*[!.]*\s*$", re.IGNORECASE),
    re.compile(r"^\s*how\s+are\s+you\??\s*$", re.IGNORECASE),
    re.compile(r"^\s*thanks?(\s+you)?\s*[!.]*\s*$", re.IGNORECASE),
    re.compile(r"^\s*(bye|goodbye|see\s+you)\s*[!.]*\s*$", re.IGNORECASE),
    re.compile(r"^\s*who\s+are\s+you\??\s*$", re.IGNORECASE),
    re.compile(r"^\s*what\s+can\s+you\s+help\s+me\s+with\??\s*$", re.IGNORECASE),
]

CHITCHAT_RESPONSES = {
    "greeting": "Hello! I'm AI StudyMate. Ask me anything about Machine Learning or Deep Learning.",
    "how_are_you": "I'm doing well, thanks for asking! What ML or DL topic can I help you with?",
    "thanks": "You're welcome! Let me know if you have another ML or DL question.",
    "farewell": "Goodbye! Come back anytime you have an ML or DL question.",
    "who_are_you": "I'm AI StudyMate, a RAG assistant that answers Machine Learning and Deep Learning questions using a verified academic corpus.",
    "what_can_you_help_with": "I can answer questions about Machine Learning and Deep Learning topics such as regression, classification, clustering, neural networks, CNNs, RNNs, and more, grounded in a verified set of academic sources.",
}

def match_chitchat_category(text):
    normalized = text.strip()
    if CHITCHAT_PATTERNS[0].match(normalized) or CHITCHAT_PATTERNS[1].match(normalized):
        return "greeting"
    if CHITCHAT_PATTERNS[2].match(normalized):
        return "how_are_you"
    if CHITCHAT_PATTERNS[3].match(normalized):
        return "thanks"
    if CHITCHAT_PATTERNS[4].match(normalized):
        return "farewell"
    if CHITCHAT_PATTERNS[5].match(normalized):
        return "who_are_you"
    if CHITCHAT_PATTERNS[6].match(normalized):
        return "what_can_you_help_with"
    return None

def is_chitchat(text):
    return match_chitchat_category(text) is not None

def chitchat_response(text):
    category = match_chitchat_category(text)
    return CHITCHAT_RESPONSES.get(category, CHITCHAT_RESPONSES["greeting"])


In [22]:
INTENT_CLASSIFIER_SYSTEM_PROMPT = """You are an intent classifier for a study assistant that only answers questions about Artificial Intelligence, Machine Learning, and Deep Learning: supervised/unsupervised learning, regression, classification, clustering, decision trees, SVM, k-NN, Naive Bayes, ensemble methods, model evaluation, overfitting/underfitting, feature selection, neural networks, backpropagation, CNNs, RNNs, LSTMs, autoencoders, optimization, regularization, attention, Transformers, GANs, VAEs, diffusion models.

Classify the user's message into exactly one label:
IN_SCOPE - a genuine question about one of the AI/ML/DL topics above.
OUT_OF_SCOPE - about anything else (cooking, sports, politics, travel, movies, general programming unrelated to AI/ML/DL, etc).

Respond with exactly one word: IN_SCOPE or OUT_OF_SCOPE. Do not explain."""

def classify_intent(user_input):
    if is_chitchat(user_input):
        return "chitchat"
    try:
        response = ollama.chat(
            model=OLLAMA_MODEL_NAME,
            messages=[
                {"role": "system", "content": INTENT_CLASSIFIER_SYSTEM_PROMPT},
                {"role": "user", "content": user_input},
            ],
            options={"temperature": 0},
        )
        label = response["message"]["content"].strip().upper()
    except Exception:
        label = "IN_SCOPE"
    if "OUT_OF_SCOPE" in label:
        return "out_of_scope"
    return "in_scope"


In [23]:
STRICT_SYSTEM_PROMPT = """You are AI StudyMate, an assistant that answers Machine Learning and Deep Learning questions using ONLY the numbered sources the user provides in their message.
Rules:
1. Use ONLY the information in the numbered sources below the question. Do not use any outside knowledge.
2. Every factual claim must be supported by one of the numbered sources. Cite it using its bracket ID exactly as given to you, for example [S1] or [S2]. Never invent an ID, and never cite an ID that was not given to you.
3. Do not use any other citation format. Do not write author names, page numbers, or titles yourself. Only use the bracket ID.
4. If a numbered source contains a partial or indirect explanation of the mechanism being asked about, use it and cite it rather than refusing. Only refuse if none of the numbered sources contain information relevant to the question.
5. If, after applying rule 4, the sources still do not support an answer, respond exactly: "AI_STUDYMATE_REFUSAL"
6. Keep answers concise and direct. Cite only the specific source IDs that support each claim, not every source you were given. Do not greet the user or ask for clarification.
7. Don't mention the page number or the section in your answer. Only cite the source ID(s) that support your answer.

"""

def build_source_map(retrieved_chunks):
    source_map = {}
    for i, chunk in enumerate(retrieved_chunks, start=1):
        source_id = f"S{i}"
        meta = chunk["metadata"]
        source_map[source_id] = {
            "label": f"{meta['title']}, page {meta['page']}, Section: {meta['section']}",
            "chunk": chunk,
        }
    return source_map

def build_prompt(question, source_map):
    context_blocks = []
    for source_id, entry in source_map.items():
        context_blocks.append(f"[{source_id}] {entry['label']}\n{entry['chunk']['text']}")
    context = "\n\n---\n\n".join(context_blocks)
    return f"Sources:\n{context}\n\nQuestion: {question}\n\nAnswer:"


In [24]:
CITATION_ID_PATTERN = re.compile(r"\[S(\d+)\]")

def extract_cited_ids(answer):
    return sorted(set(int(m) for m in CITATION_ID_PATTERN.findall(answer)))

def validate_and_render_citations(answer, source_map):
    valid_ids = set(int(source_id[1:]) for source_id in source_map.keys())
    cited_ids = extract_cited_ids(answer)
    if not cited_ids:
        return None, "no_citation_found"
    invalid_ids = [cid for cid in cited_ids if cid not in valid_ids]
    if invalid_ids:
        return None, f"invalid_citation_ids: {invalid_ids}"
    rendered = answer
    for source_id, entry in source_map.items():
        rendered = rendered.replace(f"[{source_id}]", f"({entry['label']})")
    return rendered, "valid"


In [25]:
retrieval_smoke_test_questions = [
    "What is the bias-variance tradeoff?",
    "How does dropout prevent overfitting?",
    "What is the attention mechanism in Transformers?",
]

retrieval_smoke_test_records = []
for question in retrieval_smoke_test_questions:
    retrieved = hybrid_retrieve(question)
    retrieval_smoke_test_records.append({
        "question": question,
        "retrieved_titles": [r["metadata"]["title"] for r in retrieved],
    })

pd.DataFrame(retrieval_smoke_test_records)


,question,retrieved_titles
0,What is the bias-variance tradeoff?,"[CS229 Bias-Variance Analysis, CS229 Bias-Vari..."
1,How does dropout prevent overfitting?,[Dropout: A Simple Way to Prevent Neural Netwo...
2,What is the attention mechanism in Transformers?,"[Attention Is All You Need, Attention Is All Y..."


## 2.5 End-to-End RAG

In [26]:
def call_ollama(prompt, model=OLLAMA_MODEL_NAME):
    try:
        response = ollama.chat(
            model=model,
            messages=[
                {"role": "system", "content": STRICT_SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ],
            options={"temperature": LLM_TEMPERATURE},
        )
        return response["message"]["content"]
    except Exception as e:
        return f"OLLAMA_CALL_FAILED: {e}"

def ask(question, top_k=FINAL_TOP_K):
    intent = classify_intent(question)

    if intent == "chitchat":
        return {
            "question": question,
            "intent": intent,
            "answer": chitchat_response(question),
            "is_refusal": False,
            "refusal_reason": None,
            "citation_status": "n/a (chitchat)",
            "retrieved_sources": [],
            "retrieved_pages": [],
            "retrieved_source_ids": [],
            "retrieved_chunks": [],
        }

    if intent == "out_of_scope":
        return {
            "question": question,
            "intent": intent,
            "answer": OUT_OF_SCOPE_TEXT,
            "is_refusal": True,
            "refusal_reason": "out_of_scope",
            "citation_status": "n/a (out_of_scope)",
            "retrieved_sources": [],
            "retrieved_pages": [],
            "retrieved_source_ids": [],
            "retrieved_chunks": [],
        }

    retrieved = hybrid_retrieve(question, top_k=top_k)
    source_map = build_source_map(retrieved)
    prompt = build_prompt(question, source_map)
    raw_answer = call_ollama(prompt)

    if raw_answer.strip() == "AI_STUDYMATE_REFUSAL" or raw_answer.startswith("OLLAMA_CALL_FAILED"):
        final_answer = REFUSAL_TEXT
        is_refusal = True
        refusal_reason = "llm_declined"
        citation_status = "n/a (refusal)"
    else:
        rendered, citation_status = validate_and_render_citations(raw_answer, source_map)
        if rendered is None:
            final_answer = REFUSAL_TEXT
            is_refusal = True
            refusal_reason = "citation_rejected"
        else:
            final_answer = rendered
            is_refusal = False
            refusal_reason = None

    retrieved_sources = [
        f"{r['metadata']['title']} (page {r['metadata']['page']}, {r['metadata']['section']})"
        for r in retrieved
    ]
    retrieved_pages = [r["metadata"]["page"] for r in retrieved]
    retrieved_source_ids = [r["metadata"]["source_id"] for r in retrieved]

    return {
        "question": question,
        "intent": intent,
        "answer": final_answer,
        "is_refusal": is_refusal,
        "refusal_reason": refusal_reason,
        "citation_status": citation_status,
        "retrieved_sources": retrieved_sources,
        "retrieved_pages": retrieved_pages,
        "retrieved_source_ids": retrieved_source_ids,
        "retrieved_chunks": retrieved,
    }


In [27]:
end_to_end_demo_inputs = [
    "Hello",
    "What is the capital of France?",
    "Explain how the Adam optimizer works.",
]

end_to_end_demo_results = [ask(q) for q in end_to_end_demo_inputs]
pd.DataFrame([
    {
        "question": r["question"],
        "intent": r["intent"],
        "retrieved_sources": "; ".join(r["retrieved_sources"]),
        "answer": r["answer"],
    }
    for r in end_to_end_demo_results
])


,question,intent,retrieved_sources,answer
0,Hello,chitchat,,Hello! I'm AI StudyMate. Ask me anything about...
1,What is the capital of France?,out_of_scope,,I only answer questions about Artificial Intel...
2,Explain how the Adam optimizer works.,in_scope,Adam: A Method for Stochastic Optimization (pa...,According to (Adam: A Method for Stochastic Op...


## 2.6 Evaluation

In [36]:
evaluation_questions = [
    "What is the difference between supervised and unsupervised learning?",
    "What is linear regression used for?",
    "How does logistic regression classify data?",
    "How does k-means assign points to clusters?",
    "How does a decision tree split data?",
    "How does a Support Vector Machine find a boundary?",
    "How does k-Nearest Neighbors classify a point?",
    "What assumption does Naive Bayes make?",
    "How does Random Forest use decision trees?",
    "What metrics evaluate a classification model?",
    "What is overfitting?",
    "What is feature selection?",
    "What is a neural network?",
    "How does backpropagation update weights?",
    "Why are CNNs good for image data?",
    "Why are RNNs useful for sequential data?",
    "How do LSTMs stop vanishing gradients using gates?",
    "What is an autoencoder used for?",
    "How is Adam optimizer different from SGD?",
    "How does dropout prevent overfitting?",
]

EXPECTED_PRIMARY_SOURCE_IDS = [
    ["cs229_notes1_supervised_learning", "cs229_notes7a_unsupervised"],
    ["cs229_notes1_supervised_learning"],
    ["cs229_notes1_supervised_learning", "cs229_notes2_generative_learning"],
    ["cs229_notes7a_unsupervised"],
    ["random_forests"],
    ["cs229_notes3_svm"],
    ["knn_classifiers_tutorial"],
    ["cs229_notes2_generative_learning"],
    ["random_forests"],
    ["feature_selection", "cs229_bias_variance"],
    ["cs229_bias_variance", "dropout"],
    ["feature_selection", "knn_classifiers_tutorial"],
    [],
    [],
    ["resnet"],
    ["lstm_search_space_odyssey"],
    ["lstm_search_space_odyssey"],
    ["vae"],
    ["adam_optimizer"],
    ["dropout"],
]

CHITCHAT_TEST_CASES = ["Hello", "How are you?", "Thanks"]
OUT_OF_SCOPE_TEST_CASES = [
    "How do I cook pasta?",
    "What is the capital of France?",
    "Who won a football match?",
]

len(evaluation_questions) == len(EXPECTED_PRIMARY_SOURCE_IDS)


True

In [37]:
def tokenize_for_overlap(text):
    stopwords = {
        "the", "a", "an", "is", "are", "of", "to", "in", "on", "for", "and",
        "or", "that", "this", "it", "as", "by", "with", "be", "was", "were",
        "does", "do", "how", "what", "which", "from", "at", "each",
    }
    words = re.findall(r"[a-zA-Z0-9]+", text.lower())
    return {w for w in words if w not in stopwords and len(w) > 2}

def compute_grounding_overlap(answer, retrieved_chunks):
    answer_words = tokenize_for_overlap(answer)
    if not answer_words:
        return 0.0
    context_words = set()
    for chunk in retrieved_chunks:
        context_words |= tokenize_for_overlap(chunk["text"])
    if not context_words:
        return 0.0
    overlap = answer_words & context_words
    return round(len(overlap) / len(answer_words), 3)


In [38]:
def run_routing_test(text, expected_intent):
    result = ask(text)
    actual_intent = result["intent"]
    if expected_intent == "chitchat":
        correct = actual_intent == "chitchat"
    else:
        correct = actual_intent == "out_of_scope" and result["answer"].strip() == OUT_OF_SCOPE_TEXT
    return {
        "Question": text,
        "Intent": actual_intent,
        "Expected Source": "n/a",
        "Retrieved Sources": "n/a (no retrieval)",
        "Answer": result["answer"],
        "Citation Valid": "n/a",
        "Grounded": "n/a",
        "Correct": "Correct" if correct else f"Incorrect (routed as {actual_intent}, expected {expected_intent})",
        "Notes": "routing test",
    }

routing_test_rows = []
for text in CHITCHAT_TEST_CASES:
    routing_test_rows.append(run_routing_test(text, "chitchat"))
for text in OUT_OF_SCOPE_TEST_CASES:
    routing_test_rows.append(run_routing_test(text, "out_of_scope"))

pd.DataFrame(routing_test_rows)


,Question,Intent,Expected Source,Retrieved Sources,Answer,Citation Valid,Grounded,Correct,Notes
0,Hello,chitchat,n/a,n/a (no retrieval),Hello! I'm AI StudyMate. Ask me anything about...,n/a,n/a,Correct,routing test
1,How are you?,chitchat,n/a,n/a (no retrieval),"I'm doing well, thanks for asking! What ML or ...",n/a,n/a,Correct,routing test
2,Thanks,chitchat,n/a,n/a (no retrieval),You're welcome! Let me know if you have anothe...,n/a,n/a,Correct,routing test
3,How do I cook pasta?,out_of_scope,n/a,n/a (no retrieval),I only answer questions about Artificial Intel...,n/a,n/a,Correct,routing test
4,What is the capital of France?,out_of_scope,n/a,n/a (no retrieval),I only answer questions about Artificial Intel...,n/a,n/a,Correct,routing test
5,Who won a football match?,out_of_scope,n/a,n/a (no retrieval),I only answer questions about Artificial Intel...,n/a,n/a,Correct,routing test


In [39]:
in_scope_rows = []
for i, question in enumerate(evaluation_questions):
    result = ask(question)
    expected_ids = EXPECTED_PRIMARY_SOURCE_IDS[i]
    expected_label = "; ".join(expected_ids) if expected_ids else "No dedicated source (documented gap)"

    if not expected_ids:
        retrieval_status = "documented_gap"
    elif any(sid in expected_ids for sid in result["retrieved_source_ids"]):
        retrieval_status = "hit"
    else:
        retrieval_status = "miss"

    citation_valid = result["citation_status"] == "valid"
    grounded = (not result["is_refusal"]) and citation_valid

    if result["is_refusal"]:
        if retrieval_status == "documented_gap":
            correct = "Correct (expected refusal - documented corpus gap)"
        elif retrieval_status == "miss":
            correct = "Correct (expected refusal - retrieval miss)"
        else:
            correct = "Needs review (refused despite retrieval hit)"
        citation_display = "n/a (refusal)"
    else:
        grounding_score = compute_grounding_overlap(result["answer"], result["retrieved_chunks"])
        correct = f"Manual review required (grounding_overlap={grounding_score})"
        citation_display = citation_valid

    notes = ""
    if result.get("refusal_reason") == "citation_rejected":
        notes = f"Rejected by citation validator ({result['citation_status']}), converted to refusal"

    in_scope_rows.append({
        "Question": result["question"],
        "Intent": result["intent"],
        "Expected Source": expected_label,
        "Retrieved Sources": "; ".join(sorted(set(
            c["metadata"]["title"] for c in result["retrieved_chunks"]
        ))),
        "Answer": result["answer"],
        "Citation Valid": citation_display,
        "Grounded": grounded,
        "Correct": correct,
        "Notes": notes,
    })

pd.DataFrame(in_scope_rows).drop(columns=["Notes"]).head(20)


,Question,Intent,Expected Source,Retrieved Sources,Answer,Citation Valid,Grounded,Correct
0,What is the difference between supervised and ...,in_scope,cs229_notes1_supervised_learning; cs229_notes7...,An Introduction to Variable and Feature Select...,(CS229 Lecture Notes - Supervised Learning (Li...,True,True,Manual review required (grounding_overlap=0.6)
1,What is linear regression used for?,in_scope,cs229_notes1_supervised_learning,An Introduction to Variable and Feature Select...,(CS229 Lecture Notes - Support Vector Machines...,True,True,Manual review required (grounding_overlap=0.5)
2,How does logistic regression classify data?,in_scope,cs229_notes1_supervised_learning; cs229_notes2...,CS229 Lecture Notes - Deep Learning (Neural Ne...,(CS229 Section Notes - Evaluation Metrics (Cla...,True,True,Manual review required (grounding_overlap=0.235)
3,How does k-means assign points to clusters?,in_scope,cs229_notes7a_unsupervised,CS229 Lecture Notes - Unsupervised Learning (k...,(CS229 Lecture Notes - Unsupervised Learning (...,True,True,Manual review required (grounding_overlap=0.385)
4,How does a decision tree split data?,in_scope,random_forests,Random Forests,"(Random Forests, page 2, Section: 1.1 Introduc...",True,True,Manual review required (grounding_overlap=0.5)
5,How does a Support Vector Machine find a bound...,in_scope,cs229_notes3_svm,CS229 Lecture Notes - Support Vector Machines,(CS229 Lecture Notes - Support Vector Machines...,True,True,Manual review required (grounding_overlap=0.583)
6,How does k-Nearest Neighbors classify a point?,in_scope,knn_classifiers_tutorial,k-Nearest Neighbour Classifiers: 2nd Edition (...,I don't have enough information in my verified...,n/a (refusal),False,Needs review (refused despite retrieval hit)
7,What assumption does Naive Bayes make?,in_scope,cs229_notes2_generative_learning,CS229 Lecture Notes - Generative Learning Algo...,(CS229 Lecture Notes - Generative Learning Alg...,True,True,Manual review required (grounding_overlap=0.5)
8,How does Random Forest use decision trees?,in_scope,random_forests,Random Forests,"(Random Forests, page 21, Section: EX,Y(Y−h(X))2)",True,True,Manual review required (grounding_overlap=0.75)
9,What metrics evaluate a classification model?,in_scope,feature_selection; cs229_bias_variance,CS229 Section Notes - Evaluation Metrics (Clas...,(CS229 Section Notes - Evaluation Metrics (Cla...,True,True,Manual review required (grounding_overlap=0.25)


In [40]:
evaluation_df = pd.DataFrame(routing_test_rows + in_scope_rows)
evaluation_csv_path = PROCESSED_DIR / "evaluation_results.csv"
evaluation_df.to_csv(evaluation_csv_path, index=False)
print(evaluation_csv_path)
evaluation_df


c:\Users\omara\Downloads\ITI_FinalP\data\processed\evaluation_results.csv


,Question,Intent,Expected Source,Retrieved Sources,Answer,Citation Valid,Grounded,Correct,Notes
0,Hello,chitchat,n/a,n/a (no retrieval),Hello! I'm AI StudyMate. Ask me anything about...,n/a,n/a,Correct,routing test
1,How are you?,chitchat,n/a,n/a (no retrieval),"I'm doing well, thanks for asking! What ML or ...",n/a,n/a,Correct,routing test
2,Thanks,chitchat,n/a,n/a (no retrieval),You're welcome! Let me know if you have anothe...,n/a,n/a,Correct,routing test
3,How do I cook pasta?,out_of_scope,n/a,n/a (no retrieval),I only answer questions about Artificial Intel...,n/a,n/a,Correct,routing test
4,What is the capital of France?,out_of_scope,n/a,n/a (no retrieval),I only answer questions about Artificial Intel...,n/a,n/a,Correct,routing test
5,Who won a football match?,out_of_scope,n/a,n/a (no retrieval),I only answer questions about Artificial Intel...,n/a,n/a,Correct,routing test
6,What is the difference between supervised and ...,in_scope,cs229_notes1_supervised_learning; cs229_notes7...,An Introduction to Variable and Feature Select...,(CS229 Lecture Notes - Supervised Learning (Li...,True,True,Manual review required (grounding_overlap=0.6),
7,What is linear regression used for?,in_scope,cs229_notes1_supervised_learning,An Introduction to Variable and Feature Select...,(CS229 Lecture Notes - Support Vector Machines...,True,True,Manual review required (grounding_overlap=0.5),
8,How does logistic regression classify data?,in_scope,cs229_notes1_supervised_learning; cs229_notes2...,CS229 Lecture Notes - Deep Learning (Neural Ne...,(CS229 Section Notes - Evaluation Metrics (Cla...,True,True,Manual review required (grounding_overlap=0.235),
9,How does k-means assign points to clusters?,in_scope,cs229_notes7a_unsupervised,CS229 Lecture Notes - Unsupervised Learning (k...,(CS229 Lecture Notes - Unsupervised Learning (...,True,True,Manual review required (grounding_overlap=0.385),


In [41]:
in_scope_df = evaluation_df[evaluation_df["Intent"] == "in_scope"]
routing_df = evaluation_df[evaluation_df["Intent"] != "in_scope"]

grounded_count = int(in_scope_df["Grounded"].apply(lambda v: v is True).sum())
refused_count = int((~in_scope_df["Grounded"].apply(lambda v: v is True)).sum())
citation_valid_count = int(in_scope_df["Citation Valid"].apply(lambda v: v is True).sum())
routing_correct_count = int(routing_df["Correct"].str.startswith("Correct").sum())
routing_total = len(routing_df)

summary_df = pd.DataFrame([{
    "in_scope_questions": len(in_scope_df),
    "grounded_answers": grounded_count,
    "refused_answers": refused_count,
    "citations_valid": citation_valid_count,
    "routing_tests_total": routing_total,
    "routing_tests_correct": routing_correct_count,
}])
summary_df


,in_scope_questions,grounded_answers,refused_answers,citations_valid,routing_tests_total,routing_tests_correct
0,20,19,1,19,6,6


In [42]:
failure_cases_df = evaluation_df[
    evaluation_df["Correct"].str.startswith("Incorrect")
    | evaluation_df["Correct"].str.startswith("Needs review")
][["Question", "Intent", "Retrieved Sources", "Correct", "Notes", "Answer"]]
failure_cases_df


,Question,Intent,Retrieved Sources,Correct,Notes,Answer
12,How does k-Nearest Neighbors classify a point?,in_scope,k-Nearest Neighbour Classifiers: 2nd Edition (...,Needs review (refused despite retrieval hit),Rejected by citation validator (no_citation_fo...,I don't have enough information in my verified...


## 2.7 Export

In [43]:
export_config = {
    "embedding_model": EMBEDDING_MODEL_NAME,
    "tokenizer": TOKENIZER_NAME,
    "chunk_size_tokens": TARGET_CHUNK_TOKENS,
    "chunk_overlap_tokens": OVERLAP_TOKENS,
    "min_chunk_tokens": MIN_CHUNK_TOKENS,
    "semantic_breakpoint_percentile": SEMANTIC_BREAKPOINT_PERCENTILE,
    "drop_fragment_tokens": DROP_FRAGMENT_TOKENS,
    "final_top_k": FINAL_TOP_K,
    "source_pool_size": SOURCE_POOL_SIZE,
    "fused_candidate_pool": FUSED_CANDIDATE_POOL,
    "rrf_k": RRF_K,
    "retrieval_method": "hybrid_dense_bm25_rrf_cosine_rerank",
    "chunking_method": "structure_aware_semantic_breakpoint",
    "llm_temperature": LLM_TEMPERATURE,
    "citation_format": "deterministic_source_ids",
    "intent_routing": "chitchat_regex_plus_llm_classifier",
    "ollama_model": OLLAMA_MODEL_NAME,
    "collection_name": COLLECTION_NAME,
    "vector_store_path": str(VECTOR_STORE_DIR),
    "chunk_count": len(all_chunks),
}

config_path = VECTOR_STORE_DIR / "config.json"
with open(config_path, "w") as f:
    json.dump(export_config, f, indent=2)

print(config_path)
print(collection.count())


c:\Users\omara\Downloads\ITI_FinalP\data\vector_store\config.json
654
